In [ ]:
!pip install tensorflow kagglehub

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [ ]:
import os
path
file_path = path + "/" + os.listdir(path)[0]
file_path

'/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import accuracy_score

In [ ]:
data = pd.read_csv(file_path)
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
# d = pd.read_csv(file_path)
# d.head()

In [ ]:
data['label'] = data['sentiment'].apply(lambda x: 1 if x == "positive" else 0)
data.head()

,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


In [ ]:
data.duplicated().sum()

np.int64(418)

In [ ]:
df = data.drop(['sentiment'], axis=1)
df = df.drop_duplicates()
df.head()

,review,label
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
import re

def clean_html(text):
    text = re.sub(r'<[^>]*>', '', str(text))  # Remove HTML tags
    text = re.sub(r'&[a-zA-Z0-9#]+;', ' ', text)  # Remove HTML entities
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    return text.strip()

df['review'] = df['review'].apply(clean_html)
df.head()


,review,label
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. The filming tec...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


# Stopword removal

In [ ]:
import spacy

# Download the correct medium-sized English model
spacy.cli.download("en_core_web_sm")

# Load the downloaded model
nlp = spacy.load("en_core_web_sm")
nlp

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
PAD, UNK = "<pad>", "<unk>"
# Negations carry sentiment ("not good"), so keep them even though spaCy marks them as stop words.
KEEP_WORDS = ["not", "no", "never", "nor", "n't", "cannot", "without"]

In [ ]:
text = "This is a sample sentence showing how to remove stopwords using spaCy."
text = "The movie was not good"
doc = nlp(text)

# Filter out tokens where token.is_stop is True
filtered_tokens = [(token.text) for token in doc if not token.is_stop]

print(filtered_tokens)

['movie', 'good']


In [ ]:
stop_words = nlp.Defaults.stop_words
# Use set subtraction to remove KEEP_WORDS from the stop_words set
stop_words = list(set(stop_words) - set(KEEP_WORDS))
len(stop_words)

319

In [ ]:
def remove_stopwords(text):
    doc = nlp(text)
    filtered_tokens = [token.text for token in doc if token.text.lower() not in stop_words]
    return " ".join(filtered_tokens)

In [ ]:
# stop_words = nlp.Defaults.stop_words

# print(stop_words)

In [ ]:
type(stop_words)

list

In [ ]:
len(list(stop_words))

319

In [ ]:
df['review'] = df['review'].apply(lambda x: ' '.join(word.lower() for word in x.split() if word not in stop_words))
df.head()

,review,label
0,one reviewers mentioned watching 1 oz episode ...,1
1,a wonderful little production. the filming tec...,1
2,i thought wonderful way spend time hot summer ...,1
3,basically there's family little boy (jake) thi...,0
4,"petter mattei's ""love time money"" visually stu...",1


Remove URLs

In [ ]:
import builtins
import re

# Restore the built-in str function in case it was overwritten
str = builtins.str

df['review'] = df['review'].apply(
    lambda x: re.sub(r'http\S+|www\S+', '', str(x))
)

df.head()

,review,label
0,one reviewers mentioned watching 1 oz episode ...,1
1,a wonderful little production. the filming tec...,1
2,i thought wonderful way spend time hot summer ...,1
3,basically there's family little boy (jake) thi...,0
4,"petter mattei's ""love time money"" visually stu...",1


# Train Test Split

In [ ]:
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)
train_data.head()

,review,label
7837,"i liked movie 'the emporer's new groove', watc...",0
4814,"i decided watch movie noted ""scariest movie ev...",0
35458,"it's hard going ""the lady shanghai"" film witho...",1
3446,this sci-fi adventure not best no means worst....,0
24478,"around late 1970's, animator don bluth, frustr...",0


# Tokenize and padding sequence

In [ ]:
tokenizer = Tokenizer(num_words=5000)
x_t = tokenizer.fit_on_texts(train_data['review'])
x_train = pad_sequences(tokenizer.texts_to_sequences(train_data['review']), maxlen=200)
x_test = pad_sequences(tokenizer.texts_to_sequences(test_data['review']), maxlen=200)
x_train

array([[   0,    0,    0, ...,  234,   93,  128],
       [   0,    0,    0, ..., 3109, 4202, 2642],
       [ 108, 3125,  167, ..., 2956,  295,  871],
       ...,
       [  89,    3,  309, ...,  291,  337,   63],
       [   0,    0,    0, ...,  150,  601,   26],
       [   0,    0,    0, ...,   11, 1667, 2839]],
      shape=(39665, 200), dtype=int32)

In [ ]:
print(x_t)

None


# Prepare the Target Variable

In [ ]:
y_train = train_data['label']
y_test = test_data['label']

# Build and Train LSTM model

In [ ]:
model = Sequential()

model.add(Embedding(input_dim=5000, output_dim=128, input_shape=(200,)))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 128)       │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 771,713 (2.94 MB)

 Trainable params: 771,713 (2.94 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(x_train, y_train, batch_size=64, epochs=5, validation_split=0.2)

Epoch 1/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 364s 727ms/step - accuracy: 0.8314 - loss: 0.3852 - val_accuracy: 0.8721 - val_loss: 0.3029
Epoch 2/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 378s 762ms/step - accuracy: 0.8894 - loss: 0.2775 - val_accuracy: 0.8641 - val_loss: 0.3225
Epoch 3/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 364s 734ms/step - accuracy: 0.9081 - loss: 0.2297 - val_accuracy: 0.8718 - val_loss: 0.2993
Epoch 4/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 356s 718ms/step - accuracy: 0.9243 - loss: 0.1953 - val_accuracy: 0.8748 - val_loss: 0.3105
Epoch 5/5
496/496 ━━━━━━━━━━━━━━━━━━━━ 378s 709ms/step - accuracy: 0.9358 - loss: 0.1660 - val_accuracy: 0.8709 - val_loss: 0.3530


# Sentiment Analysis using logistic regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score